# Deploy a Gemma 4 Model on OpenShift AI

This notebook guides you through deploying a **Gemma 4** model on OpenShift AI using a custom vLLM ServingRuntime. Once deployed, you can evaluate the model using the LMEvalJob notebooks in this workshop.

## Overview

We will:
1. Register a custom vLLM ServingRuntime with Gemma 4 support
2. Deploy the model via the **OpenShift AI Dashboard**
3. Verify the deployment

## Prerequisites

- OpenShift AI cluster with GPU nodes (NVIDIA)
- `oc` CLI logged in with admin or project-level privileges
- Hugging Face access to `google/gemma-4-E2B-it` (accept license on HF)

## Step 1: Configuration

In [25]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "gemma4-e2b-deployment")
HF_MODEL_ID = os.getenv("HF_MODEL_ID", "google/gemma-4-E2B-it")
RUNTIME_NAME = os.getenv("RUNTIME_NAME", "vllm-cuda-runtime-gemma4")
HF_TOKEN_SECRET = os.getenv("HF_TOKEN_SECRET", "hf-token")

print(f"Namespace:        {NAMESPACE}")
print(f"Model:            {MODEL_NAME}")
print(f"HF Model:         {HF_MODEL_ID}")
print(f"Runtime:          {RUNTIME_NAME}")
print(f"HF Token Secret:  {HF_TOKEN_SECRET}")

Namespace:        hyo-project
Model:            gemma4-e2b-deployment
HF Model:         google/gemma-4-E2B-it
Runtime:          vllm-cuda-runtime-gemma4
HF Token Secret:  hf-token


## Step 2: Verify Cluster Access

Make sure you are logged in to the OpenShift cluster. If not, run:

```bash
oc login --server=<your-cluster-api-url> --web
```

In [5]:
!oc whoami
!oc project {NAMESPACE}

admin
Already on project "hyo-project" on server "https://api.ocp.d5j7k.sandbox3050.opentlc.com:6443".


## Step 3: Register Custom ServingRuntime

OpenShift AI ships with default ServingRuntimes, but Gemma 4 requires a preview vLLM image with Gemma 4 support. We register a custom runtime that uses `registry.redhat.io/rhaii-preview/vllm-cuda-rhel9:gemma4`.

Register via the **OpenShift AI Dashboard**:
1. Navigate to **Settings > Serving runtimes**
2. Click **Add serving runtime**
3. Select **Start from scratch** and paste the YAML generated below
4. Click **Add**

![Custom ServingRuntime Registration](../images/serving-runtime-registration.png)

The cell below generates the YAML to copy into the dashboard:

In [3]:
yaml_runtime = f"""apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  annotations:
    opendatahub.io/recommended-accelerators: '["nvidia.com/gpu"]'
    opendatahub.io/runtime-version: v0.13.0
    openshift.io/display-name: vLLM NVIDIA GPU ServingRuntime for KServe (Gemma4)
    opendatahub.io/apiProtocol: REST
  labels:
    opendatahub.io/dashboard: "true"
  name: {RUNTIME_NAME}
spec:
  annotations:
    opendatahub.io/kserve-runtime: vllm
    prometheus.io/path: /metrics
    prometheus.io/port: "8080"
  containers:
    - args:
        - --port=8080
        - --model=/mnt/models
        - --served-model-name={{{{.Name}}}}
      command:
        - python
        - -m
        - vllm.entrypoints.openai.api_server
      env:
        - name: HF_HOME
          value: /tmp/hf_home
      image: registry.redhat.io/rhaii-preview/vllm-cuda-rhel9:gemma4
      name: kserve-container
      ports:
        - containerPort: 8080
          protocol: TCP
  multiModel: false
  supportedModelFormats:
    - autoSelect: true
      name: vLLM
"""

print("Copy the YAML below and paste it into the OpenShift AI Dashboard:")
print("=" * 60)
print(yaml_runtime)

Copy the YAML below and paste it into the OpenShift AI Dashboard:
apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  annotations:
    opendatahub.io/recommended-accelerators: '["nvidia.com/gpu"]'
    opendatahub.io/runtime-version: v0.13.0
    openshift.io/display-name: vLLM NVIDIA GPU ServingRuntime for KServe (Gemma4)
    opendatahub.io/apiProtocol: REST
  labels:
    opendatahub.io/dashboard: "true"
  name: vllm-cuda-runtime-gemma4
spec:
  annotations:
    opendatahub.io/kserve-runtime: vllm
    prometheus.io/path: /metrics
    prometheus.io/port: "8080"
  containers:
    - args:
        - --port=8080
        - --model=/mnt/models
        - --served-model-name={{.Name}}
      command:
        - python
        - -m
        - vllm.entrypoints.openai.api_server
      env:
        - name: HF_HOME
          value: /tmp/hf_home
      image: registry.redhat.io/rhaii-preview/vllm-cuda-rhel9:gemma4
      name: kserve-container
      ports:
        - containerPort: 8080
 

## Step 4: Create HF Token Secret for Model Download

Gemma 4 is a gated model on Hugging Face. The ServingRuntime needs access to download it:

In [ ]:
!oc get secret {HF_TOKEN_SECRET} -n {NAMESPACE} 2>/dev/null || \
    echo "Run 1_LMEval_setup.ipynb first to create {HF_TOKEN_SECRET} secret"

NAME       TYPE     DATA   AGE
hf-token   Opaque   1      7h20m


## Step 5: Deploy the Model via OpenShift AI Dashboard

Deploy through the dashboard — it automatically creates the namespace-scoped ServingRuntime, InferenceService, TLS certificates, and OAuth proxy (kube-rbac-proxy sidecar).

1. Navigate to **Model Serving > Deploy model**
2. Configure the following:

| Field | Value |
|-------|-------|
| **Model deployment name** | e.g., `gemma4-e2b-deployment` (this becomes the `served_model_name`) |
| **Serving runtime** | Select the Gemma4 runtime registered in Step 3 |
| **Model framework** | vLLM |
| **Model location (URI)** | `hf://google/gemma-4-E2B-it` |

3. Under **Additional serving runtime arguments**, add each line as a separate argument:

```
--model=/mnt/models
--tensor-parallel-size=1
--enable-auto-tool-choice
--reasoning-parser=gemma4
--tool-call-parser=gemma4
--max-model-len=32768
--gpu-memory-utilization=0.90
--language-model-only
```

4. Set **resources** to match your GPU hardware (e.g., 8 CPU, 128Gi memory, 1 GPU)
5. Click **Deploy**

![Model Deployment](../images/model-deploy.png)

> **Important:** After deploying, update `MODEL_NAME` in your `.env` to match the deployment name you entered in the dashboard.

In [26]:
print("After deploying from the dashboard, verify the InferenceService:")
!oc get inferenceservice {MODEL_NAME} -n {NAMESPACE}

After deploying from the dashboard, verify the InferenceService:
NAME                    URL                                                                                READY   PREV   LATEST   PREVROLLEDOUTREVISION   LATESTREADYREVISION   AGE
gemma4-e2b-deployment   https://gemma4-e2b-deployment-hyo-project.apps.ocp.d5j7k.sandbox3050.opentlc.com   True                                                                  68m


In [27]:
print("Verify the ServingRuntime created by the dashboard:")
!oc get servingruntime -n {NAMESPACE} | grep -E "NAME|{MODEL_NAME}"

Verify the ServingRuntime created by the dashboard:
NAME                          DISABLED   MODELTYPE   CONTAINERS         AGE
gemma4-e2b-deployment                    vLLM        kserve-container   68m


## Step 6: Wait for Model to be Ready

The model download and initialization takes a few minutes (typically 3-5 min for Gemma 4 E2B):

In [28]:
import subprocess, time

print(f"Waiting for inferenceservice/{MODEL_NAME} to be ready...")
for i in range(30):
    result = subprocess.run(
        ["oc", "get", "inferenceservice", MODEL_NAME, "-n", NAMESPACE,
         "-o", "jsonpath={.status.conditions[?(@.type==\"Ready\")].status}"],
        capture_output=True, text=True
    )
    status = result.stdout.strip() or "Pending"
    print(f"  {status}  ({(i+1)*30}s elapsed)")
    if status == "True":
        print("Model is ready!")
        break
    time.sleep(30)
else:
    print("Timed out. Check pod status with: oc get pods -n", NAMESPACE)


Waiting for inferenceservice/gemma4-e2b-deployment to be ready...
  True  (30s elapsed)
Model is ready!


## Step 7: Verify Deployment

In [29]:
!oc get inferenceservice {MODEL_NAME} -n {NAMESPACE}
print("\n--- Deployment pods ---")
!oc get pods -n {NAMESPACE} | grep {MODEL_NAME}

NAME                    URL                                                                                READY   PREV   LATEST   PREVROLLEDOUTREVISION   LATESTREADYREVISION   AGE
gemma4-e2b-deployment   https://gemma4-e2b-deployment-hyo-project.apps.ocp.d5j7k.sandbox3050.opentlc.com   True                                                                  68m

--- Deployment pods ---
gemma4-e2b-deployment-predictor-6b7ccdf57-pjnvw   2/2     Running                  0             23m


## Step 8: Confirm served_model_name

The `served_model_name` is automatically set to the InferenceService name. Verify in vLLM logs:

In [31]:
!oc logs deployment/{MODEL_NAME}-predictor -n {NAMESPACE} 2>&1 | grep served_model_name | head -1

SERVED_MODEL_NAME = MODEL_NAME
print(f"\nserved_model_name = {SERVED_MODEL_NAME}")
print(f"Use this as the 'model' value in LMEvalJob and API requests.")

(APIServer pid=1) INFO 05-07 08:25:02 [utils.py:233] non-default args: {'enable_auto_tool_choice': True, 'tool_call_parser': 'gemma4', 'port': 8080, 'model': '/mnt/models', 'max_model_len': 32768, 'served_model_name': ['gemma4-e2b-deployment'], 'reasoning_parser': 'gemma4', 'language_model_only': True}

served_model_name = gemma4-e2b-deployment
Use this as the 'model' value in LMEvalJob and API requests.


## Key Configuration Notes

| Setting | Value | Purpose |
|---------|-------|--------|
| `served_model_name` | Dashboard deployment name (= `MODEL_NAME` in `.env`) | Used as `model` in LMEvalJob |
| Deploy method | OpenShift AI Dashboard (Single-model serving) | Creates ServingRuntime + InferenceService + OAuth proxy |
| OAuth auth | Enabled automatically by dashboard | Requires SA token for API access |
| Port | `8443` (OAuth proxy) / `8080` (vLLM) | LMEvalJob connects to 8443 |
| Internal URL | `https://<name>-predictor.<ns>.svc.cluster.local:8443` | Used as `base_url` |
| Image | `registry.redhat.io/rhaii-preview/vllm-cuda-rhel9:gemma4` | vLLM with Gemma 4 support |

## Done!

Your Gemma 4 model is now deployed and ready for evaluation. Proceed to:
- **1_LMEval_setup.ipynb** — Configure RBAC and secrets for LMEvalJob
- **2_eval_hub_setup.ipynb** — Configure EvalHub SDK with MLflow experiment tracking
- **1_builtin_tasks/** — Run evaluations with built-in tasks
- **2_custom_tasks/** — Run full Korean benchmarks